# ZTE — Sub-word (token) level

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/victor-iyi/zte/blob/main/notebooks/alignments/zte_token.ipynb)

One of three notebooks that differ in **which unit the contrastive term pulls at**, and nothing else.

| Level | The unit it aligns | Frozen target |
| --- | --- | --- |
| `sentence` | the pooled sentence vector | a frozen E5 sentence embedding |
| `word` | one fixated word = one EEG token | a frozen word vector |
| `token` | four fixed intra-word slices of one word | the LM's sub-word embeddings |

**This notebook is the `token` level.** This is the level the whitepaper's future-work list puts first, and the only one that could change the ceiling rather than the error bars. Sentence InfoNCE pulls at the pooled vector and the word term at whole words; neither ever asks a slice of one word's EEG to mean the word-piece it spells. Frequency-matched word retrieval sits at **0.99x chance over 107,962 queries**, so lexical structure was never requested rather than refuted.

Every arm here is byte-identical to `experiments/ablation/exp16_residual_off.yaml` — the programme's
best-measured recipe — except the two weights that switch this level on. Read the configuration diff, not the
run names.

### How to read a number in this project

1. **`scoreboard.held_out_retrieval`, never `sentence_retrieval`.** The pooled figure is computed over the
   training subjects too, so it rewards memorising the brains you have rather than reaching the one you do not.
2. **Top-k is a hit count out of 700 with an exact binomial tail.** At chance 1/700, Top-1 expects exactly one
   hit; "0.006 vs 0.001" is three hits at *p* ≈ 0.08.
3. **Rank percentile leads**, because it uses every query rather than only the winners.
4. **A single seed is never a headline.** Retrieval on this project does not reproduce across retrainings within
   a factor of three; geometry does, to within 1.03–1.58.
5. **Every number carries the floor it must clear** — sentence length for the sentence level, and the sub-word
   piece profile for the token level.

## 1 · Provision the runtime

Installs `uv`, clones or refreshes the repo, and builds the pinned Python 3.14 venv every `!uv run` below uses. The kernel you are typing in is Colab's own older interpreter and never imports `zte`.

In [ ]:
%%bash
pip install -q uv
# Work whether this is a fresh runtime (/content), a re-run already inside zte/, or a restored session.
if [ -f pyproject.toml ]; then :
elif [ -d zte/.git ]; then cd zte
else git clone --depth 1 https://github.com/victor-iyi/zte.git --branch main && cd zte
fi
git fetch --depth 1 origin main && git reset --hard FETCH_HEAD
echo "ZTE @ $(git rev-parse --short HEAD): $(git log -1 --pretty=%s)"
uv python install 3.14
uv sync --all-groups

## 2 · Wire the kernel

`colab()` is this notebook's only route into ZTE: it runs one `zte-colab` subcommand in the venv and returns the JSON it printed.

In [ ]:
import json
import os
import platform
import subprocess
from typing import Any


def colab(command: str, *args: str) -> dict[str, Any]:
    """Runs one `zte-colab` subcommand in the provisioned venv and returns the JSON object it printed.

    This is the notebook's only route into ZTE. The package runs on 3.14 inside the uv venv; this kernel is
    Colab's own older interpreter, so it renders payloads rather than computing them.
    """
    argv = ['uv', 'run', 'zte-colab', command, *args]
    done = subprocess.run(argv, capture_output=True, text=True, check=False)
    if done.returncode != 0:
        raise RuntimeError(f'`{" ".join(argv)}` failed:\n{done.stderr[-3000:]}')

    return json.loads(done.stdout)


# Enter the repo in the notebook kernel, so relative paths and every subprocess resolve. A %%bash `cd` cannot
# do this: it dies with its own shell.
if os.path.isdir('zte') and not os.path.isfile('pyproject.toml'):
    os.chdir('zte')

ENV = colab('env')
os.environ.update(ENV['env'])

try:
    from google.colab import userdata  # type: ignore[import-untyped]

    _hf = userdata.get('HF_TOKEN')
except Exception as exc:  # not on Colab, or the secret is not granted to this notebook
    _hf, _ = None, print(f'HF_TOKEN unavailable ({type(exc).__name__}) — Hub downloads will be unauthenticated.')
if _hf:
    os.environ['HF_TOKEN'] = _hf
    print('HF_TOKEN loaded — authenticated HuggingFace Hub downloads enabled.')

print(f'repo   : {ENV["root"]}')
print(f'venv   : Python {ENV["venv"]["python"]} · zte {ENV["venv"]["zte"]}   ← every `!uv run` command')
print(f'kernel : Python {platform.python_version()}   ← this cell; renders payloads, never imports zte')
print(f'env    : {", ".join(ENV["env"])}')

## 3 · What hardware did you get

The campaign below assumes an A100. On a smaller card, drop `train.batch_size` to 64 and raise `grad_accum_steps` to 2 — but note that halves the in-batch negatives a contrastive objective depends on, so a batch-64 arm is **not** matched to a batch-128 one and must say so.

In [ ]:
ENV = colab('env')
plan, res = ENV['plan'], ENV['resources']

print(f'backend   : {plan["backend"]}  ·  device {plan["device"]}  ·  autocast {plan["autocast_dtype"]}')
print(f'workers   : {plan["dataloader_workers_auto"]}  ·  pin_memory {plan["pin_memory"]}')
print(f'resources : {res["ram_gb"]} GB RAM · {res["cpu_count"]} cores · {res["free_disk_gb"]} GB free')
print(f'gpu       : {res.get("gpu") or "none — this campaign needs one"}')
if not res.get('gpu'):
    print('\nNo accelerator. Runtime -> Change runtime type -> GPU before training anything.')

## 4 · Drive is the workspace

**Nothing of value is written only to the VM disk.** Checkpoints mirror to Drive after every epoch, evaluation lands straight on Drive, and a reclaimed VM resumes from there. Mount first.

In [ ]:
from google.colab import drive  # type: ignore[import-untyped]

drive.mount('/gdrive')

In [ ]:
# Set to an existing folder name (e.g. '2026-08-13') to resume that session; None starts today's.
RESUME_DATE: str | None = None
# 'local+mirror' trains on the VM disk and copies to Drive after each stage (recommended).
# 'drive' writes runs straight to Drive: slower, but nothing to mirror if the VM dies mid-epoch.
WRITE_MODE: str = 'local+mirror'
ZTE_DRIVE: str = '/gdrive/My Drive/Sharables/ZTE'

_resume = ('--resume-date', RESUME_DATE) if RESUME_DATE else ()
SESSION = colab('session', '--drive', ZTE_DRIVE, '--write-mode', WRITE_MODE, *_resume)

# Every `!` command below inherits these, so the bundle cache, the data root and the backup target are wired once.
os.environ.update(SESSION['env'])

RUN_DATE: str = SESSION['run_date']
DATA_DIR: str = SESSION['data_dir']
DRIVE_DIR: str = SESSION['session_dir']
DRIVE_RUNS: str = SESSION['drive_runs']
DRIVE_ANALYSIS: str = SESSION['drive_analysis']
LOCAL_RUNS: str = SESSION['local_runs']
OUT_ROOT: str = SESSION['out_root']
DRIVE_BACKUP: str = SESSION['drive_backup']
PREPARED_LOCAL: str = SESSION['prepared_local']
PREPARED_DRIVE: str = SESSION['prepared_drive']

print(f'session   : {RUN_DATE}   ({"resumed" if SESSION["resumed"] else "new"})')
print(f'Drive     : {SESSION["drive_root"]}   (mounted: {SESSION["drive_mounted"]})')
print(f'raw data  : {DATA_DIR}   (present: {SESSION["data_dir_present"]})')
print(f'runs ->   : {OUT_ROOT}   (backed up to {DRIVE_BACKUP})')
print(f'analysis  : {DRIVE_ANALYSIS}')
print(f'prepared  : {PREPARED_DRIVE}   (staged on the VM at {PREPARED_LOCAL})')

### 4a · Helpers this notebook uses everywhere

In [ ]:
import pathlib


def find_runs(*extra: str, headline: bool = False) -> list[dict[str, Any]]:
    """Every run reachable right now — each dated Drive session newest first, then the local disk.

    A run is anything with a `config.yaml`, so one a reclaimed VM killed mid-training is still listed; `evaluated`
    is what says whether it got as far as producing numbers.
    """
    roots = [*extra, LOCAL_RUNS]
    flags = ['--headline'] if headline else []

    return colab('runs', '--drive', ZTE_DRIVE, '--experiments', *roots, *flags)['runs']


def every_session() -> list[str]:
    """Every dated session's run folder on Drive, newest first — what the analysis section reads across."""
    return colab('runs', '--drive', ZTE_DRIVE)['sessions']


def resolve_ckpt(run_name: str, which: str = 'best') -> str:
    """Finds a run's checkpoint, Drive first, so a fresh VM can decode a session it did not train.

    A missing `best.pt` never falls back to `last.pt`: they are different models, and swapping them silently
    misattributes the number.
    """
    for run in colab('runs', '--drive', ZTE_DRIVE, '--experiments', LOCAL_RUNS, '--run', run_name)['runs']:
        if path := run['checkpoints'][which]:
            print(f'{which}.pt for {run_name}: {"Drive" if run["source"] == "drive" else "local disk"}\n  {path}')
            return path

    raise FileNotFoundError(f'no {which}.pt for {run_name!r} on Drive or locally; train it first (Section 7).')


def durable(*parts: str) -> str:
    """A path under the durable root: this session's Drive folder on Colab, `res/` on a machine without it.

    Everything expensive that does *not* resume -- the analysis dashboard, the studio page, the rebaseline audit --
    is written here rather than written locally and copied later, so a VM reclaimed during the *next* cell cannot
    take it.
    """
    root = DRIVE_DIR if SESSION['drive_mounted'] else 'res'
    path = os.path.join(root, *parts)
    os.makedirs(os.path.dirname(path) or path, exist_ok=True)

    return path


def _mirror(direction: str, date: str, sub: str, local: str | None) -> None:
    """Runs one mirror and reports what moved, or why nothing did."""
    where = ['--drive', ZTE_DRIVE, '--write-mode', WRITE_MODE, *(('--local', local) if local else ())]
    payload = colab('mirror', *where, '--direction', direction, '--date', date, '--sub', sub)

    if reason := payload['skipped_reason']:
        print(f'nothing mirrored: {reason}')
        return

    print(f'{payload["src"]} -> {payload["dst"]}   ({payload["copied"]} copied, {payload["failed"]} failed)')


def mirror_to_drive(local: str | None = None, sub: str = 'experiments') -> None:
    """Copy the VM's runs to Drive, minus what is rebuildable, so the session survives the machine."""
    _mirror('up', RUN_DATE, sub, local)


def restore_from_drive(run_date: str | None = None, sub: str = 'experiments', local: str | None = None) -> None:
    """Pull a session's runs back to the VM so every `--resume` finds its work after a runtime reset."""
    _mirror('down', run_date or RUN_DATE, sub, local)


def show_resources() -> None:
    """Prints RAM / GPU / disk as they stand now, so an out-of-memory kill is predictable rather than a mystery."""
    res = colab('env')['resources']
    gpu = f'{res["gpu"]["name"]} ({res["gpu"]["total_gb"]} GB)' if res['gpu'] else 'none'
    print(f'RAM {res["ram_gb"]} GB · {res["cpu_count"]} cores · {res["free_disk_gb"]} GB free disk · GPU {gpu}')


show_resources()

In [ ]:
import datetime


def archive_to_drive(note: str | None = None) -> None:
    """Provenance-stamped zip of the best checkpoints, skipping synthetic smoke runs."""
    stamp = datetime.datetime.now().strftime('%H%M%S')
    out = f'{DRIVE_DIR}/archives/zte_{RUN_DATE}_{stamp}.zip'
    command = ['uv', 'run', 'zte-pack', 'zip', '--all', '--best-only', '--skip-synthetic', '--out', out]
    subprocess.run([*command, *(['--note', note] if note else [])], check=False)
    print('archive ->', out)


def snapshot_to_drive(note: str | None = None) -> str:
    """Everything -- runs, cache, benchmark, explorer -- in one zip, so a local session never re-prepares data."""
    stamp = datetime.datetime.now().strftime('%H%M%S')
    out = f'{DRIVE_DIR}/archives/zte_snapshot_{RUN_DATE}_{stamp}.zip'
    command = ['uv', 'run', 'zte-pack', 'snapshot', '--skip-synthetic', '--out', out]
    subprocess.run([*command, *(['--note', note] if note else [])], check=False)
    print('snapshot ->', out)

    return out

In [ ]:
# Rendering only: these read CSV and figure JSON that `zte-colab` already produced.
import pandas as pd
import plotly.io as pio

## 5 · Prepare the data once, on Drive, and never again

The processed bundle is content-addressed by the dataset config and published to `Sharables/ZTE/prepared`. On a cache hit this is ~15 seconds and the multi-GB `.mat` extraction is skipped entirely. The levels share their bundles: `objective.*` is outside the dataset cache key, so all three notebooks reuse the same four bundles -- one per task set (SR+NR, NR, SR, TSR), because `dataset.tasks` *is* inside it.

In [ ]:
!uv run zte-prepare --root "{DATA_DIR}" --configs experiments/alignment --cache-dir "{PREPARED_LOCAL}" --cache-remote "{PREPARED_DRIVE}"

## 6 · The `token` level

**Unit:** four fixed intra-word slices of one word · **Target:** the frozen LM's sub-word embeddings

This is the level the whitepaper's future-work list puts first, and the only one that could change the ceiling rather than the error bars. Sentence InfoNCE pulls at the pooled vector and the word term at whole words; neither ever asks a slice of one word's EEG to mean the word-piece it spells. Frequency-matched word retrieval sits at **0.99x chance over 107,962 queries**, so lexical structure was never requested rather than refuted.

The four arms of this level, and the single lever that separates it from its siblings:

```
experiments/alignment/token/combined.yaml   SR + NR together
experiments/alignment/token/nr.yaml         normal reading alone
experiments/alignment/token/sr.yaml         sentiment reading alone
experiments/alignment/token/tsr.yaml        task-specific reading alone
```

In [ ]:
LEVEL = 'token'
ARMS = ['combined', 'nr', 'sr', 'tsr']
CONFIGS = {arm: f'experiments/alignment/{LEVEL}/{arm}.yaml' for arm in ARMS}

for arm, path in CONFIGS.items():
    print(f'{arm:<9} {path}')

In [ ]:
# The single-lever diff between this level and the other two, read from the files rather than from the names.
KNOBS = ('lexical_weight', 'lexical_reader_weight', 'token_weight', 'token_reader_weight')


def levers(path: str) -> dict[str, str]:
    """The four alignment weights as written in one config."""
    out = {}
    for line in pathlib.Path(path).read_text().splitlines():
        key = line.strip().split(':')[0]
        if key in KNOBS:
            out[key] = line.strip()
    return out


for other in ('sentence', 'word', 'token'):
    mine, theirs = levers(CONFIGS['combined']), levers(f'experiments/alignment/{other}/combined.yaml')
    moved = [k for k in KNOBS if mine.get(k) != theirs.get(k)]
    print(f'{LEVEL:>9} vs {other:<9} moves: {moved or "nothing (same level)"}')

### 6a · The gate this level must clear — read before quoting any number

The sub-word piece profile is a **brain-free channel larger than the sentence identity it would be used to
recover**. Measured with the real `Qwen/Qwen2.5-0.5B` tokeniser on a 700-sentence corpus matched to ZuCo's
statistics (1.463 pieces per word against ZuCo's measured 1.4):

| What an oracle is told, and nothing else | Bits | Top-1 | hits / 700 |
| --- | ---: | ---: | ---: |
| `n_words` — the documented 5.14-bit length confound | 4.96 | 4.71% | 33 |
| **total sub-word pieces — one integer per sentence** | **5.58** | **8.86%** | **62** |
| `(n_words, total pieces)` jointly | 8.18 | 51.29% | 359 |
| **the per-word piece profile** | **9.44** | **99.57%** | **697** |
| the same profile after ZuCo's 33% word omission | 9.37 | 96.14% | 673 |

$H(\text{identity}) = 9.4512$ bits on a 700-sentence gallery, so the ordered profile leaves 0.01 bits
unresolved. **A single integer retrieves 62 of 700 where the best encoder in this programme retrieves 26.**

**And there is a second channel, in the substrate rather than the level.** The per-word window is a
variable-length fixation zero-padded to 350 samples and then z-scored across the whole padded width, so its tail
is an *exactly constant* value beginning at sample $L$ — the fixation length. That is an eye-tracking quantity
word length is readable from, so the boundary hands every raw-representation encoder a per-word length estimate
for free. Slot $k$ is supervised only for words with more than $k$ pieces, and piece count moves with word length,
so this level adds an incentive to use it. No structural guard can see the difference, because neither route
carries a reference tensor. §10 runs the piece oracle; the padding ablation has not been run at all.

Two things follow, both enforced in code rather than by discipline:

- `token_sub_tokens` is a **fixed 4 for every word**, never the number of pieces its reference spells it in. The
  piece count enters the loss's target mask and nothing the encoder computes, so a reading is encoded identically
  whatever sentence it turns out to be. `tests/test_token_alignment.py` allows the intra-word path exactly two
  callers, both training-side; a third fails the suite.
- Every headline from this notebook is gated on `zte-rebaseline --piece-oracle`, run in §10. A number below the
  worst oracle is reported as **not evidence of decoding**, exactly as an unstratified Top-k is for length.

## 7 · The campaign

54 runs, ~109 GPU-hours across all three levels, in three tiers. **Every stopping point is a complete,
reportable table**, so a reclaimed VM or a spent budget still leaves something publishable.

| Tier | What it answers | Runs (all levels) |
| --- | --- | ---: |
| `mechanism` | 3 levels × 2 regimes, holdout ZAB, seed 42 — comparable to the published table | 12 |
| `power` | 3 levels × combined × all 12 folds — the population estimate with a real CI | 36 |
| `spread` | 3 levels × combined × ZAB × seeds 42/43/44 — seed noise apart from fold noise | 6 |

A run counts as **done** when its `evaluation/metrics.json` exists on Drive. Re-running this notebook re-plans
and skips finished work, so it is safe to run it as many times as the VM lifetime forces.

In [ ]:
PLAN = colab('sweep', 'plan', '--levels', LEVEL, '--out-root', OUT_ROOT, '--drive', ZTE_DRIVE)
STATUS = colab('sweep', 'status', '--levels', LEVEL, '--out-root', OUT_ROOT, '--drive', ZTE_DRIVE)

progress = STATUS['progress']

print(f'{LEVEL} level: {PLAN["planned"]} runs, {PLAN["hours"]:.1f} GPU-hours')
for block in progress['tiers']:
    print(
        f'  {block["tier"]:<11} {block["done"]:>3}/{block["total"]:<3} done  ·  {block["hours_remaining"]:.1f} h left'
    )
print(f'  {"TOTAL":<11} {progress["done"]:>3}/{progress["total"]:<3} done  ·  {progress["hours_remaining"]:.1f} h left')

In [ ]:
frame = pd.DataFrame(STATUS['runs'])
display(frame[['tier', 'regime', 'task', 'holdout', 'seed', 'run_name', 'done']])

### 7a · Train — the mechanism tier

Four arms: SR+NR together, then each reading task alone so §8's transfer matrix has its vantage points. Every
cell passes `--resume`, which is idempotent: a finished run exits in seconds, an interrupted one continues from
its last epoch, and **omitting it would reseed `best.pt` on Drive at epoch 1**.

These arms carry `eval_profile: sweep`. Evaluation, not training, is two thirds of a run on this project's
measured timings; the sweep profile keeps exactly the numbers that are allowed to be a headline and drops the
figures and the interactive explorers. Run the full profile on a winning arm in §10.

In [ ]:
HOLDOUT = 'ZAB'
SEED = 42

for arm in ARMS:
    name = f'align_{LEVEL}_{arm}_lo{HOLDOUT}_s{SEED}'
    print(f'\n===== {name} =====')
    !uv run zte-run --config "{CONFIGS[arm]}" --root "{DATA_DIR}" --name "{name}" \
        --out-root "{OUT_ROOT}" --loso-holdout {HOLDOUT} --seed {SEED} \
        --data-cache "{PREPARED_LOCAL}" --drive-backup "{DRIVE_BACKUP}" --spatial exact --resume

mirror_to_drive()

### 7b · The power tier — twelve strangers

Twelve folds of the combined arm at one seed. This is what turns a single-participant number into a population
estimate with a spread. A previous twelve-fold sweep found retrieval varying by a factor of 85 between the
easiest and hardest held-out brain, uncorrelated with how much data that person contributed — so one fold is an
anecdote, however carefully it is measured.

**This is the long cell.** ~25 GPU-hours per level. It is fully resumable; run it, lose the VM, run it again.

In [ ]:
FOLDS = ['ZAB', 'ZDM', 'ZGW', 'ZJM', 'ZJN', 'ZJS', 'ZKB', 'ZKH', 'ZKW', 'ZMG', 'ZPH', 'ZDN']

for fold in FOLDS:
    name = f'align_{LEVEL}_combined_lo{fold}_s42'
    print(f'\n===== {name} =====')
    !uv run zte-run --config "{CONFIGS['combined']}" --root "{DATA_DIR}" --name "{name}" \
        --out-root "{OUT_ROOT}" --loso-holdout {fold} --seed 42 \
        --data-cache "{PREPARED_LOCAL}" --drive-backup "{DRIVE_BACKUP}" --spatial exact --resume
    mirror_to_drive()

# Named one by one: `zte-loso-summary` keys folds on the holdout alone, so pointing it at the whole run root
# would average this level's twelve folds together with the single-task arms and any sibling level's runs.
FOLD_DIRS = ' '.join(f'"{OUT_ROOT}/align_{LEVEL}_combined_lo{fold}_s42"' for fold in FOLDS)

!uv run zte-loso-summary --experiments {FOLD_DIRS} --out "{DRIVE_ANALYSIS}/LOSO_{LEVEL}.md

### 7c · The spread tier — the error bars a reviewer will ask for

Three seeds of the combined arm on one fold. Held-out retrieval on this project spans a factor of 2.86 between seeds of an unchanged configuration and 3.0 between retrainings four days apart, so this is not optional garnish — it is what makes the mechanism-tier comparison readable at all.

In [ ]:
for seed in (42, 43, 44):
    name = f'align_{LEVEL}_combined_lo{HOLDOUT}_s{seed}'
    print(f'\n===== {name} =====')
    !uv run zte-run --config "{CONFIGS['combined']}" --root "{DATA_DIR}" --name "{name}" \
        --out-root "{OUT_ROOT}" --loso-holdout {HOLDOUT} --seed {seed} \
        --data-cache "{PREPARED_LOCAL}" --drive-backup "{DRIVE_BACKUP}" --spatial exact --resume

mirror_to_drive()

## 8 · Parallax — the transfer matrix

The strongest single piece of evidence this project has. A model trained on **normal reading** and evaluated on
**sentiment reading** faces a passage set it has never seen, on a subject it has never seen. If the effect were
passage memorisation, that cell would collapse. At the sentence level it does not: 0.9623 rank percentile on
novel passages against 0.9530 in-task.

This section asks the same question of the `token` level. Each cell is one trained arm scored on another
task's readings.

In [ ]:
TASKS = ['NR', 'SR', 'TSR']
TRANSFERS = f'{DRIVE_ANALYSIS}/parallax_{LEVEL}'

for train_task in TASKS:
    ckpt = resolve_ckpt(f'align_{LEVEL}_{train_task.lower()}_lo{HOLDOUT}_s{SEED}')
    for eval_task in TASKS:
        print(f'\n----- train {train_task} -> eval {eval_task} -----')
        !uv run zte-parallax transfer --ckpt "{ckpt}" --eval-task {eval_task} \
            --root "{DATA_DIR}" --out "{TRANSFERS}" --seed {SEED}

!uv run zte-parallax report --transfers "{TRANSFERS}" --out "{DRIVE_ANALYSIS}/parallax_{LEVEL}_report"

In [ ]:
import pandas as pd

MATRIX = colab('panels', '--experiments', TRANSFERS, '--out', f'{DRIVE_ANALYSIS}/panels_{LEVEL}', '--only', 'transfer')
for panel in MATRIX['panels']:
    print(f'{panel["name"]}: {panel["path"]}')

## 9 · Decode — one token at a time, against seven controls

Generation is scored **strictly autoregressively**: teacher forcing is off, decoding is greedy, and no
ground-truth prefix exists anywhere in the scored path. A teacher-forced perplexity is computed as a diagnostic
and quarantined from the verdict.

The control battery is the point. The replication literature has shown that published ZuCo BLEU is produced
under teacher forcing, and that a model fed **pure noise scores higher** than one fed EEG. So every decode below
runs its own Gaussian-noise control, and it is printed beside the headline whichever way it lands.

The decoder-rescored retrieval number is reported as **retrieval, never as generation**.

In [ ]:
DECODE_CFG = 'experiments/decoder/decode_parallax_nr.yaml'
ENCODER = resolve_ckpt(f'align_{LEVEL}_nr_lo{HOLDOUT}_s{SEED}')
DECODER = f'decode_align_{LEVEL}_nr_lo{HOLDOUT}_s{SEED}'

!uv run zte-run --config "{DECODE_CFG}" --root "{DATA_DIR}" --name "{DECODER}" \
    --out-root "{OUT_ROOT}" --encoder-ckpt "{ENCODER}" --seed {SEED} \
    --data-cache "{PREPARED_LOCAL}" --drive-backup "{DRIVE_BACKUP}" --resume

mirror_to_drive()

In [ ]:
DECODE_OUT = f'{DRIVE_ANALYSIS}/decode_{LEVEL}'

!uv run zte-decode --ckpt "{resolve_ckpt(DECODER)}" --root "{DATA_DIR}" --split test \
    --controls mean_prefix,null_prefix,phase,noise,shuffled_z,length_only,mismatch \
    --seeds 42,43,44 --within-task SR,NR --capacity --out "{DECODE_OUT}"

In [ ]:
READINGS = colab('readings', '--from', DECODE_OUT, '--rows', '12')
verdict = READINGS['verdict']

print('above_controls :', verdict.get('above_controls'))
for clause, value in (verdict.get('clauses') or {}).items():
    print(f'  {clause:<34} {value}')
print('\nA False anywhere above means no generation number from this run is a headline. That is the expected')
print('result at this bit budget and it is reported plainly rather than dropped.')

## 10 · Analysis — and the floor this level must clear

`zte-rebaseline` re-scores the trained checkpoint against the confounds it had free access to. It retrains
nothing and it gates nothing automatically — it hands you the floor, and the number is yours to read against it.

**For this level the floor is the sub-word piece profile**, which is worth 9.4 bits on a 700-sentence gallery. `--piece-oracle` scores all four signatures and returns a verdict.

In [ ]:
AUDIT = f'{DRIVE_ANALYSIS}/rebaseline_{LEVEL}'
# Bound on its own line: IPython's `!` formatter cannot parse a nested brace and silently leaves the whole
# command untransformed, so `--out "{AUDIT}"` would become a literal directory name on the VM disk.
COMBINED_CKPT = resolve_ckpt(f'align_{LEVEL}_combined_lo{HOLDOUT}_s{SEED}')

!uv run zte-rebaseline --ckpt "{COMBINED_CKPT}" \
    --root "{DATA_DIR}" --out "{AUDIT}" --piece-oracle

In [ ]:
report = json.loads(pathlib.Path(f'{AUDIT}/rebaseline.json').read_text())

floor = report.get('floor_comparison') or {}
print(f'held-out rank percentile : {floor.get("encoder")}   (CI low {floor.get("encoder_ci_low")})')
print(f'length oracle at +/-{floor.get("oracle_tol")}      : {floor.get("oracle")}')
print(f'clears the length floor  : {floor.get("clears_floor")}')

if piece := report.get('piece_oracle'):
    print('\n--- sub-word piece oracles ---')
    for name, block in piece['oracles'].items():
        hits = block['top1'] * block['n']
        print(
            f'  {name:<10} Top-1 {block["top1"]:.4f}  ({hits:.0f}/{block["n"]:.0f})  {block["information_bits"]:.2f} bits'
        )
    print(f'\n  worst case : {piece["worst_case_signature"]} at {piece["worst_case_top1"]:.4f}')
    print(f'  this run   : {piece["observed_top1"]}')
    print(f'  VERDICT    : {piece["verdict"]}')

### 10a · The cross-level table

All three levels side by side, each with its gallery size, its chance level, its hit count out of N, an exact binomial tail and its rank percentile. A token-level row is **refused** unless it carries its piece-oracle floor.

In [ ]:
ROOTS = [*every_session(), LOCAL_RUNS]
# Joined here rather than inside the `!` line: a nested brace makes IPython abandon the interpolation entirely.
ROOT_ARGS = ' '.join(f'"{r}"' for r in ROOTS)

!uv run zte-analyze --experiments {ROOT_ARGS} --out "{DRIVE_ANALYSIS}"

In [ ]:
def table(name: str) -> pd.DataFrame:
    """One of the analysis CSVs, or an empty frame when this session has not produced it yet."""
    path = f'{DRIVE_ANALYSIS}/tables/{name}.csv'
    try:
        return pd.read_csv(path)
    except OSError:
        print(f'{name}.csv not written yet')
        return pd.DataFrame()


display(table('multi_seed'))
display(table('loso'))

### 10b · What the contrastive term bought

The alignment/uniformity decomposition, per level. **Alignment** is the mean cosine between positive pairs — how
tightly the loss pulls two readings of the same thing together. **Uniformity** is how evenly the rest of the
sphere is used. A contrastive objective that collapses scores well on the first and terribly on the second, and
the pair is what says whether the geometry is real or a cone.

Reported beside effective rank, because invariance bought by destroying capacity is not invariance.

In [ ]:
import json
import pathlib

# One pass over the trained encoder produces both the atlas and the contrastive geometry, because both read the
# same three sets of vectors -- and all three come out of `project()`, so they genuinely share one space.
ATLAS_JSON = f'{DRIVE_ANALYSIS}/atlas_{LEVEL}.json'
# A run trained with --data-cache writes no bundle and the Drive mirror excludes it, so the corpus is named.
RUN_DIR = str(pathlib.Path(resolve_ckpt(f'align_{LEVEL}_combined_lo{HOLDOUT}_s{SEED}')).parent.parent)

!uv run zte-visualize --run "{RUN_DIR}" --root "{DATA_DIR}" --kind levels --out "{ATLAS_JSON}" --max-points 4000

PAYLOAD = json.loads(pathlib.Path(ATLAS_JSON).read_text())

In [ ]:
geometry = PAYLOAD['contrastive']


def num(value, width, spec='+.4f'):
    "A number, or an em dash when the level carried too few positive pairs to score one."
    return format(value, f'>{width}{spec}') if isinstance(value, (int, float)) else '—'.rjust(width)


print(f'{"level":<10}{"alignment":>11}{"uniformity":>12}{"eff-rank":>10}  positive-negative gap (95% CI)')
for name, block in geometry['levels'].items():
    ci = block.get('positive_negative_gap_ci') or []
    interval = f'[{ci[0]:+.4f}, {ci[1]:+.4f}]' if len(ci) == 2 else 'n/a'
    excludes = block.get('gap_excludes_zero')
    print(
        f'{name:<10}{num(block.get("alignment"), 11)}{num(block.get("uniformity"), 12)}'
        f'{num(block.get("effective_rank_ratio"), 10, ".3f")}  '
        f'{num(block.get("positive_negative_gap"), 8)} {interval}'
        f'{"" if excludes is None else ("  clears zero" if excludes else "  CROSSES ZERO")}'
    )
print()
print(f'widest gap: {geometry["widest_gap"]}')
print('An interval crossing zero bought nothing. A tight alignment reached by collapsing the space is not a')
print('result either, which is why effective rank is printed beside it.')

pio.from_json(json.dumps(PAYLOAD['figures']['contrastive'])).show()

## 11 · The atlas — where token, word and sentence actually land

One projection fitted over all three levels at once, so the picture is three views of **one** geometry rather
than three unrelated scatter plots. 2D and 3D, coloured by level, hoverable down to the individual token, word
or sentence.

The explained variance is printed with it. A 2D picture of a 768-dimensional space that does not say how much it
kept is decoration.

In [ ]:
projection = PAYLOAD['projection']

print(f'projection : {PAYLOAD["method"]}, {projection["fitted_on"]}')
print(f'fitted on  : {projection["n_fit_rows"]} rows of {projection["embed_dim"]}-d vectors')
print(f'2D keeps   : {projection["explained_variance_2d"]:.1%} of the variance')
print(f'3D keeps   : {projection["explained_variance_3d"]:.1%}')
print(f'one basis  : {projection["views_share_a_basis"]}')
for block in PAYLOAD['levels']:
    print(f'  {block["level"]:<10}{block["n"]:>6} points')

pio.from_json(json.dumps(PAYLOAD['figures']['2d'])).show()
pio.from_json(json.dumps(PAYLOAD['figures']['3d'])).show()

## 12 · Persist, resume, continue

Everything above already lives on Drive. This snapshots the session so a fresh VM — or a fresh month — picks it up unchanged.

In [ ]:
mirror_to_drive()
archive_to_drive(note=f'alignment-{LEVEL}')
show_resources()

STATUS = colab('sweep', 'status', '--levels', LEVEL, '--out-root', OUT_ROOT, '--drive', ZTE_DRIVE)
progress = STATUS['progress']
for block in progress['tiers']:
    print(f'{block["tier"]:<11} {block["done"]:>3}/{block["total"]:<3} done  ·  {block["hours_remaining"]:.1f} h left')

nxt = STATUS['next']
print()
print(f'next up: {nxt["run_name"]}' if nxt else 'nothing left to run at this level.')